In [1]:
!pip -q install -U google-genai

In [2]:
from google import genai
from google.genai import types

import os, time, re, json
import pandas as pd
from pathlib import Path

In [3]:
import os
os.environ["GEMINI_API_KEY"] = "REMOVED"

In [4]:
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [5]:
# models = list(client.models.list())
# all_names = [m.name for m in models]
# print("Models visible to this key:", *all_names, sep="\n- ")

Models visible to this key:
- models/embedding-gecko-001
- models/gemini-2.5-pro-preview-03-25
- models/gemini-2.5-flash-preview-05-20
- models/gemini-2.5-flash
- models/gemini-2.5-flash-lite-preview-06-17
- models/gemini-2.5-pro-preview-05-06
- models/gemini-2.5-pro-preview-06-05
- models/gemini-2.5-pro
- models/gemini-2.0-flash-exp
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-exp-image-generation
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.0-flash-preview-image-generation
- models/gemini-2.0-flash-lite-preview-02-05
- models/gemini-2.0-flash-lite-preview
- models/gemini-2.0-pro-exp
- models/gemini-2.0-pro-exp-02-05
- models/gemini-exp-1206
- models/gemini-2.0-flash-thinking-exp-01-21
- models/gemini-2.0-flash-thinking-exp
- models/gemini-2.0-flash-thinking-exp-1219
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/learnlm-2.0-flash-experimental
- models/gemma-3-1b-it
- models

In [6]:
MODEL_NAME = "gemini-2.5-flash"

In [19]:
RESPONSE_SCHEMA = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "prompt_variants": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(type=types.Type.STRING),
            min_items=3, max_items=3,
        ),
        "response_variants": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(type=types.Type.STRING),
            min_items=3, max_items=3,
        ),
    },
    required=["prompt_variants", "response_variants"],
)


In [20]:
# JSON extractioN
def _extract_json_text(resp) -> str:
    # Try the quick accessor
    t = getattr(resp, "text", None)
    if t and t.strip():
        return t.strip()

    # Walk parts
    if getattr(resp, "candidates", None):
        for cand in resp.candidates:
            parts = getattr(getattr(cand, "content", None), "parts", []) or []
            for p in parts:
                if getattr(p, "text", None):
                    return p.text.strip()

    return ""

def _parse_json_loose(txt: str):
    if not txt:
        raise ValueError("empty response")

    # code fence block ```json ... ```
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", txt, re.I)
    if m:
        return json.loads(m.group(1).strip())

    # first balanced {...} block
    s = txt
    start = s.find("{")
    if start != -1:
        depth, in_str, esc = 0, False, False
        for i in range(start, len(s)):
            ch = s[i]
            if in_str:
                if esc:
                    esc = False
                elif ch == "\\":
                    esc = True
                elif ch == '"':
                    in_str = False
            else:
                if ch == '"':
                    in_str = True
                elif ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        candidate = s[start:i+1]
                        return json.loads(candidate)
    # C) last try: direct load (maybe it was plain JSON with whitespace)
    return json.loads(txt)

In [7]:
PROMPT_STYLES = [
    "更口语但保持专业与中立，不加入新实体，不改变原意。",
    "更正式、简洁，合并近义词，避免冗余修饰，不改变原意。",
    "更面向游客新手的问法，但不得显式提示答案，不改变原意。"
]
RESPONSE_STYLES = [
    "保留要点，用分点短句；不新增未经证实的具体数字；保留不确定性与“以官网为准”。",
    "改写为连贯段落；语气礼貌克制；不要扩写未出现的景点或价格。",
    "以‘先结论后细节’结构改写；数值/时间若不确定请明确标注不确定并提醒以官网为准。"
]
SYS_PROMPT = (
    "你是资深中文旅游数据增强助手。请严格遵守："
    "1) 只做改写，不新增事实或地点；2) 不输出任何角色标签或特殊标记；"
    "3) 输出纯文本；4) 输出为中文；5) 保持原意与事实一致。"  
    "6) 严格的长度限制：用户提问改写≤120字；回答改写≤200字，"
    "若采用列点≤6条且单条≤25字。"
)

In [8]:
# throttle
INTER_CALL_SECONDS  = 7.5  

# Output  
PROMPT_MAX_TOKENS   = 518
MAX_OUTPUT_TOKENS   = 1200

# character budgets 
PROMPT_MAX_CHARS    = 150
RESPONSE_MAX_CHARS  = 300

In [9]:
def sanitize(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r'(?:<\|image\|>\s*)+', '', text)
    text = re.sub(r'(?:>。)+', '。', text)
    text = re.sub(r'(?i)\b(?:Human:|User:|Assistant:)\b', '', text)
    text = text.replace("<|system|>", "").replace("<|user|>", "").replace("<|assistant|>", "")
    text = re.sub(r'[ \t]+\n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def _extract_text(resp) -> str:
    t = getattr(resp, "text", None)
    if t:
        return t.strip()
    # walk candidates/parts if cannot
    for cand in getattr(resp, "candidates", []) or []:
        parts = getattr(getattr(cand, "content", None), "parts", []) or []
        buf = []
        for p in parts:
            if getattr(p, "text", None):
                buf.append(p.text)
            elif isinstance(p, dict) and p.get("text"):
                buf.append(p["text"])
        if buf:
            return "\n".join(buf).strip()
    return ""

def truncate_chars(s: str, limit: int) -> str:
    s = (s or "").strip()
    if len(s) <= limit:
        return s
    cut = s[:limit]
    for mark in ["。","！","？",".","!","?"]:
        k = cut.rfind(mark)
        if k >= int(0.7 * limit):
            return cut[:k+1]
    return cut

Run three at the same time to keep within budget

In [21]:
def rewrite_both_json(client, model_name: str, prompt_text: str, response_text: str):
    task_instruction = (
        "根据以下【用户提问】与【回答】，分别按给定风格生成 3 个改写版本。"
        "约束：不改变语义与事实；不新增实体/时间/地点；不得包含任何标签或占位符；"
        f"用户提问每条≤{PROMPT_MAX_CHARS}字；回答每条≤{RESPONSE_MAX_CHARS}字。"
        "只输出符合架构的JSON，不要任何解释、注释或代码块标记。"
        "JSON字段：{\"prompt_variants\":[str,str,str], \"response_variants\":[str,str,str]}。"
    )

    payload = {
        "instruction": task_instruction,
        "styles": {
            "prompt": PROMPT_STYLES,
            "response": RESPONSE_STYLES
        },
        "data": {
            "user_prompt": sanitize(prompt_text),
            "assistant_response": sanitize(response_text)
        }
    }

    last_err = None
    for attempt in range(1, 6):
        try:
            resp = client.models.generate_content(
                model=model_name,
                contents=json.dumps(payload, ensure_ascii=False),
                config=types.GenerateContentConfig(
                    system_instruction=SYS_PROMPT,
                    temperature=0.6,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                    response_mime_type="application/json",
                    response_schema=RESPONSE_SCHEMA, # <<< hard-enforce JSON shape
                    thinking_config=types.ThinkingConfig(thinking_budget=0),  # disable thinking
                ),
            )

            txt = _extract_json_text(resp)
            obj = _parse_json_loose(txt)  # tolerate wrappers/fences
            # Normalize & truncate
            pv = [sanitize(x)[:PROMPT_MAX_CHARS]   for x in obj.get("prompt_variants", [])][:3]
            rv = [sanitize(x)[:RESPONSE_MAX_CHARS] for x in obj.get("response_variants", [])][:3]
            while len(pv) < 3: pv.append(sanitize(prompt_text)[:PROMPT_MAX_CHARS])
            while len(rv) < 3: rv.append(sanitize(response_text)[:RESPONSE_MAX_CHARS])
            return pv[:3], rv[:3]

        except Exception as e:
            last_err = e
            time.sleep(1.8 ** (attempt - 1))

    raise RuntimeError(f"rewrite_both_json failed after retries: {last_err}")

In [11]:
IN_PATH = Path("Book1.xlsx") 
df = pd.read_excel(IN_PATH)
df.head(3)

,Prompt,Response
0,请问一下成都适合家庭出游的旅游景点有哪些呢？,1. 成都大熊猫繁育研究基地：这是专门致力于大熊猫保护和研究的机构，园内大熊猫数量众多，孩子...
1,请问如果我想三日内游遍成都所有著名景点路线该如何规划？,第一天，上午前往四川博物院，这是西南最大博物馆，可重点参观巴蜀青铜馆、汉代陶石馆等，欣赏说唱...
2,请问一日游玩都江堰景区的开销一般是多少呢,一日游玩都江堰景区的开销一般在150元至350元之间，具体如下：从成都前往都江堰，若选择动车...


In [17]:
df.head(8)

,Prompt,Response
0,请问一下成都适合家庭出游的旅游景点有哪些呢？,1. 成都大熊猫繁育研究基地：这是专门致力于大熊猫保护和研究的机构，园内大熊猫数量众多，孩子...
1,请问如果我想三日内游遍成都所有著名景点路线该如何规划？,第一天，上午前往四川博物院，这是西南最大博物馆，可重点参观巴蜀青铜馆、汉代陶石馆等，欣赏说唱...
2,请问一日游玩都江堰景区的开销一般是多少呢,一日游玩都江堰景区的开销一般在150元至350元之间，具体如下：从成都前往都江堰，若选择动车...
3,请问游遍成都著名景点的比较经济的路线有哪些,第一天：市区文化探索\n 上午前往武侯祠，这里是中国唯一君臣合祀祠庙，红墙竹影拍照很美，门票...
4,请问想要在成都充分领会四川特色文化应该去哪些景点？,想要在成都充分领悟四川特色文化，可以去以下景点：\n 武侯祠 ：全国重点文物保护单位，是中国...
5,成都的最佳旅游季节是什么时候?气候如何?,成都是个一年四季都值得去的城市，但要说最佳旅游季节推荐春季和秋季，具体来说是3月–6月和9月...
6,成都必去的经典景点有哪些以及行程规划?,成都是一座“来了就不想走”的城市，其魅力源于深厚的文化底蕴、可爱的国宝和闲适的生活气息。对于...
7,成都的特色文化和地道美食有哪些?,A.特色文化\n成都的特色文化极其丰富，远不止于美食和熊猫，它是一座将闲适、雅致、市井、创新...


In [22]:
OUT_JSONL = Path("train_messages_augmented.jsonl")

In [23]:
START_ROW = 7  
mode = "a" if OUT_JSONL.exists() else "w"

In [24]:
written = 0
with OUT_JSONL.open(mode, encoding="utf-8") as f_out:
    for _, row in df.iloc[START_ROW:].iterrows():
        user0 = sanitize(str(row["Prompt"]))
        asst0 = sanitize(str(row["Response"]))
        pv, rv = rewrite_both_json(client, MODEL_NAME, user0, asst0)  # 1 call/row

        for pp in pv:
            for aa in rv:
                f_out.write(json.dumps({
                    "messages": [
                        {"role": "user", "content": pp},
                        {"role": "assistant", "content": aa},
                    ]
                }, ensure_ascii=False) + "\n")
                written += 1
        time.sleep(INTER_CALL_SECONDS)

print({"resumed_from_row": START_ROW, "new_examples_appended": written, "out": str(OUT_JSONL)})

{'resumed_from_row': 7, 'new_examples_appended': 990, 'out': 'train_messages_augmented.jsonl'}


In [25]:
from itertools import islice
with OUT_JSONL.open("r", encoding="utf-8") as f:
    for line in islice(f, 5):
        print(line.strip())

{"messages": [{"role": "user", "content": "嗨，想问问成都哪些地方比较适合带家人一起去玩呢？"}, {"role": "assistant", "content": "成都家庭游推荐：\n* 大熊猫繁育研究基地：近距离看熊猫，环境好。\n* 都江堰景区：了解古老水利工程，寓教于乐。\n* 武侯祠：感受三国文化，适合拍照。\n* 宽窄巷子：体验老成都风情，品尝小吃。\n* 锦里古街：欣赏传统建筑，看民俗表演。\n* 青城山：登山亲近自然，空气清新。"}]}
{"messages": [{"role": "user", "content": "嗨，想问问成都哪些地方比较适合带家人一起去玩呢？"}, {"role": "assistant", "content": "成都拥有多处适合家庭出游的景点。例如，成都大熊猫繁育研究基地让孩子们能近距离观察大熊猫。都江堰景区则能让全家了解古老水利工程的智慧。武侯祠可带您深入体验三国文化，并提供拍照留念的绝佳背景。宽窄巷子和锦里古街展现了老成都的生活风貌与传统文化，提供美食与民俗体验。此外，青城山以其清幽环境和适中步道，是全家登山放松的好去处。"}]}
{"messages": [{"role": "user", "content": "嗨，想问问成都哪些地方比较适合带家人一起去玩呢？"}, {"role": "assistant", "content": "结论：成都为家庭游客提供了多样化的选择，包括文化、自然和休闲类景点。\n细节：\n* 大熊猫繁育研究基地：观赏国宝，环境宜人。\n* 都江堰景区：学习水利智慧，历史悠久。\n* 武侯祠：探秘三国文化，古迹丰富。\n* 宽窄巷子：体验老成都生活，品尝特色小吃。\n* 锦里古街：感受川西风情，观赏传统技艺。\n* 青城山：登山放松，享受自然风光。"}]}
{"messages": [{"role": "user", "content": "请推荐成都适合家庭出游的旅游景点。"}, {"role": "assistant", "content": "成都家庭游推荐：\n* 大熊猫繁育研究基地：近距离看熊猫，环境好。\n* 都江堰景区：了解古老水利工程，寓教于乐。\n* 武侯祠：感受三国文化，适合拍照。\n* 宽窄巷子：体验老成都风情，品尝小吃。

In [33]:
from typing import Tuple, List, Dict, Any
import hashlib

In [35]:
BASE_JSONL = Path("train_messages.jsonl")
AUG_JSONL  = Path("train_messages_augmented.jsonl")

APPENDED_JSONL = Path("train_merged_appended.jsonl")  # raw append result
DEDUP_JSONL    = Path("train_merged_dedup.jsonl") # appended to deduped
DUPES_LOG      = Path("duplicates.jsonl") # duplicates collected
BAD_LINES_LOG  = Path("bad_lines_in_append.txt")

In [36]:
written = 0
bad = 0

with APPENDED_JSONL.open("w", encoding="utf-8") as fout, BAD_LINES_LOG.open("w", encoding="utf-8") as flog:
    for src in (BASE_JSONL, AUG_JSONL):
        if not src.exists():
            continue
        with src.open("r", encoding="utf-8") as fin:
            for line in fin:
                s = line.strip()
                if not s:
                    continue
                try:
                    json.loads(s)
                except Exception as e:
                    flog.write(f"{src.name}: {e}\n{s}\n\n")
                    bad += 1
                    continue
                fout.write(s + "\n")
                written += 1

print({"appended_written": written, "bad_lines_skipped": bad, "out": str(APPENDED_JSONL), "bad_log": str(BAD_LINES_LOG) if bad else None})

{'appended_written': 1170, 'bad_lines_skipped': 0, 'out': 'train_merged_appended.jsonl', 'bad_log': None}


In [31]:
def canonical_key(messages: List[Dict[str,str]]) -> str:
    payload = {"messages": messages}
    s = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def load_validate_jsonl(path: Path, bad_log: Path, auto_clean: bool = True):
    valids, invalids, keys = [], [], []
    if not path.exists():
        return valids, invalids, keys

    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception as e:
                invalids.append((i, f"JSON parse error: {e}", line))
                continue
            ok, reason, norm = validate_qwen_messages(obj, auto_clean=auto_clean)
            if ok:
                valids.append(norm)
                keys.append(canonical_key(norm["messages"]))
            else:
                invalids.append((i, reason, line))

    if invalids:
        with bad_log.open("w", encoding="utf-8") as f:
            for ln, why, raw in invalids:
                f.write(f"[line {ln}] {why}\n{raw}\n\n")

    return valids, invalids, keys

In [37]:
def canonical_key_from_obj(obj: dict) -> str:
    msgs = obj.get("messages", [])
    canon = {"messages": [{"role": str(m.get("role","")), "content": str(m.get("content",""))} for m in msgs]}
    s = json.dumps(canon, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

seen = set()
kept, removed = 0, 0

with APPENDED_JSONL.open("r", encoding="utf-8") as fin, \
     DEDUP_JSONL.open("w", encoding="utf-8") as fout, \
     DUPES_LOG.open("w", encoding="utf-8") as fdupes:
    for line in fin:
        s = line.strip()
        if not s:
            continue
        try:
            obj = json.loads(s)
        except Exception:
            # If a corrupted line made it through, just skip it silently here
            continue

        key = canonical_key_from_obj(obj)
        if key in seen:
            fdupes.write(json.dumps(obj, ensure_ascii=False) + "\n")
            removed += 1
            continue

        seen.add(key)
        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
        kept += 1

print({
    "dedup_kept": kept,
    "dedup_removed": removed,
    "out_dedup": str(DEDUP_JSONL),
    "dupes_log": str(DUPES_LOG) if removed else None
})

{'dedup_kept': 1170, 'dedup_removed': 0, 'out_dedup': 'train_merged_dedup.jsonl', 'dupes_log': None}


In [38]:
from itertools import islice

def preview(path: Path, n=5):
    print(f"--- {path} ---")
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(islice(f, n), start=1):
            print(f"[{i}] {line.strip()}")
    print()

preview(APPENDED_JSONL, n=3)
preview(DEDUP_JSONL, n=3)
if DUPES_LOG.exists():
    preview(DUPES_LOG, n=3)

--- train_merged_appended.jsonl ---
[1] {"messages": [{"role": "user", "content": "请问一下成都适合家庭出游的旅游景点有哪些呢？"}, {"role": "assistant", "content": "1. 成都大熊猫繁育研究基地：这是专门致力于大熊猫保护和研究的机构，园内大熊猫数量众多，孩子们可以近距离观察到大熊猫的憨态可掬，了解它们的生活习性，同时基地环境优美，适合全家一起散步游玩。 2. 都江堰景区 ：作为中华第一古堰，都江堰水利工程有着悠久的历史和伟大的智慧。家长可以带孩子参观宝瓶口引水口、飞沙堰泄洪坝、鱼嘴分水堤等，了解四六分水、二八淘沙的治水原理，寓教于乐。 3. 武侯祠 ：这里是三国迷的圣地，能让游客深入了解三国文化，园内的古建、历史遗迹、雕塑和石碑等元素丰富，观赏性和可玩性高，还可以在红墙竹影处拍摄武侠风亲子照。 4. 宽窄巷子：是成都的历史文化街8区，保留了众多老成都的生活风貌，街道相对平坦，适合带着老人和小孩漫步，大家可以在这里欣赏古建筑，品尝兔头、冒菜等地道的成都小吃。 5. 锦里古街：充满川西传统风情，街道两旁是仿古建筑，有各种特色小店和琳琅满目的小吃，能让全家人都愉快满足，还可以观看皮影戏、吹糖人等传统技艺表演。  6. 青城山：被誉为“青城天下幽”，是道教发源地之一，山清水秀，步道平缓，适合全家一起登山游玩，享受大自然的美好，呼吸新鲜空气。"}]}
[2] {"messages": [{"role": "user", "content": "请问如果我想三日内游遍成都所有著名景点路线该如何规划？"}, {"role": "assistant", "content": "第一天，上午前往四川博物院，这是西南最大博物馆，可重点参观巴蜀青铜馆、汉代陶石馆等，欣赏说唱俑等必看展品，建议游览2-3小时。中午在附近品尝成都美食，如乐山鲜知味钵钵鸡等。下午参观杜甫草堂，游览正门、大廨、诗史堂等景点，感受杜甫故居的文化氛围，建议游览2小时左右。晚上前往宽窄巷子，在这里欣赏古街风貌，品尝成都小吃，还可以选择去蜀风雅韵观看川剧变脸等表演，感受成都的夜生活。 第二天上午前往武侯祠，感受三国文化，参观红墙竹影等著名景点，建议游览1-2小时。之后游览锦里古街，体验川西特色风情，品尝